In [1]:
# The DAG Executor
# Given a task dict mapping task names to {'deps': [dep_names], 'func': lambda_source_string}, topologically order the tasks. For each task, evaluate its lambda by passing a dict {dep_name: result_of_dep} as 'inputs'. Return a dict of task_name -> result. Raise an exception if a cycle exists.
# EXAMPLE
# INPUT
# tasks
# :
# {
#   "load": {
#     "deps": ["transform"],
#     "func": "lambda inputs: sum(inputs['transform'])"
#   },
#   "extract": {"deps":[],"func":"lambda inputs: [1, 2, 3]"},
#   "transform": {
#     "deps": ["extract"],
#     "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
#   }
# }
# OUTPUT
# {"load":12,"extract":[1,2,3],"transform":[2,4,6]}


In [14]:
input_data={
  "load": {
    "deps": ["transform"],
    "func": "lambda inputs: sum(inputs['transform'])"
  },
  "extract": {"deps":[],"func":"lambda inputs: [1, 2, 3]"},
  "transform": {
    "deps": ["extract"],
    "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
  }
}

In [23]:
def execute_dag(tasks):
    results = {}
    state = {}

    def run(task_name):
        if state.get(task_name) == 'visiting':       # loop? error
            raise Exception("Cycle detected")
        if state.get(task_name) == 'done':           # already done? reuse
            return results[task_name]

        state[task_name] = 'visiting'                # mark: starting

        inputs = {}
        for dep in tasks[task_name]['deps']:         # run deps first
            inputs[dep] = run(dep)

        func = eval(tasks[task_name]['func'])        # make the function
        results[task_name] = func(inputs)            # run it, save answer
        state[task_name] = 'done'                    # mark: finished
        return results[task_name]

    for task_name in tasks:
        run(task_name)
    return results

In [24]:
execute_dag(input_data)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}

NameError: name 'input_data' is not defined

In [26]:
from collections import defaultdict, deque

# =========================
# Input
# =========================
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

# =========================
# Step 1: Validate dependencies
# =========================
for task_name, task_info in tasks.items():
    for dep in task_info.get("deps", []):
        if dep not in tasks:
            raise ValueError(f"Missing dependency '{dep}' for task '{task_name}'")

# =========================
# Step 2: Build graph
# If task depends on dep:
# dep -> task
# =========================
graph = defaultdict(list)
indegree = {}

for task_name in tasks:
    indegree[task_name] = 0

for task_name, task_info in tasks.items():
    for dep in task_info.get("deps", []):
        graph[dep].append(task_name)
        indegree[task_name] += 1

# =========================
# Step 3: Topological sort
# =========================
queue = deque()

for task_name, degree in indegree.items():
    if degree == 0:
        queue.append(task_name)

topo_order = []

while queue:
    current_task = queue.popleft()
    topo_order.append(current_task)

    for next_task in graph[current_task]:
        indegree[next_task] -= 1

        if indegree[next_task] == 0:
            queue.append(next_task)

# =========================
# Step 4: Cycle detection
# =========================
if len(topo_order) != len(tasks):
    raise ValueError("Cycle detected in task dependencies")

# =========================
# Step 5: Execute tasks
# =========================
results = {}

for task_name in topo_order:
    deps = tasks[task_name].get("deps", [])

    inputs = {}

    for dep in deps:
        inputs[dep] = results[dep]

    func = eval(tasks[task_name]["func"])

    results[task_name] = func(inputs)

# =========================
# Output
# =========================
print("Topological order:", topo_order)
print("Results:", results)

Topological order: ['extract', 'transform', 'load']
Results: {'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [28]:
tasks = {

    "load": {

        "deps": ["transform"],

        "func": "lambda inputs: sum(inputs['transform'])"

    },

    "extract": {

        "deps": [],

        "func": "lambda inputs: [1, 2, 3]"

    },

    "transform": {

        "deps": ["extract"],

        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"

    }

}

In [30]:
# store final results here
results = {}

# 1. run extract first
inputs = {}
func = eval(tasks["extract"]["func"])
print(func)
results["extract"] = func(inputs)

print(results)

<function <lambda> at 0x10bc3d280>
{'extract': [1, 2, 3]}


In [45]:
func({eval(tasks['extract']['func'])})



[1, 2, 3]

In [46]:
tasks = {
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    }
}

func_string = tasks["extract"]["func"]
print(func_string)

func = eval(func_string)

result = func({})

print(result)

lambda inputs: [1, 2, 3]
[1, 2, 3]


In [47]:
tasks = {
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    }
}

func_string = tasks["extract"]["func"]
print(func_string)

func = eval(func_string)

result = func({})

print(result)

lambda inputs: [1, 2, 3]
[1, 2, 3]


In [52]:
inputs = {
    "extract": results["extract"]
}

func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

print(results)

KeyError: 'transform'

In [54]:
tasks = {

    "load": {

        "deps": ["transform"],

        "func": "lambda inputs: sum(inputs['transform'])"

    },

    "extract": {

        "deps": [],

        "func": "lambda inputs: [1, 2, 3]"

    },

    "transform": {

        "deps": ["extract"],

        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"

    }

}

In [55]:
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

In [56]:
# store final results here
results = {}

# 1. run extract first
inputs = {}
func = eval(tasks["extract"]["func"])
results["extract"] = func(inputs)

print(results)

{'extract': [1, 2, 3]}


In [58]:
inputs = {
    "extract": results["extract"]
}

func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6]}


In [59]:
inputs = {
    "transform": results["transform"]
}

func = eval(tasks["load"]["func"])
results["load"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [60]:
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

results = {}

# Run extract
inputs = {}
func = eval(tasks["extract"]["func"])
results["extract"] = func(inputs)

# Run transform
inputs = {
    "extract": results["extract"]
}
func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

# Run load
inputs = {
    "transform": results["transform"]
}
func = eval(tasks["load"]["func"])
results["load"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [61]:
def execute_dag(tasks):
    results = {}
    state = {}

    def run(task_name):
        if state.get(task_name) == 'visiting':       # loop? error
            raise Exception("Cycle detected")
        if state.get(task_name) == 'done':           # already done? reuse
            return results[task_name]

        state[task_name] = 'visiting'                # mark: starting

        inputs = {}
        for dep in tasks[task_name]['deps']:         # run deps first
            inputs[dep] = run(dep)

        func = eval(tasks[task_name]['func'])        # make the function
        results[task_name] = func(inputs)            # run it, save answer
        state[task_name] = 'done'                    # mark: finished
        return results[task_name]

    for task_name in tasks:
        run(task_name)
    return results

In [62]:
def execute_dag(tasks):
    results = {}
    done = set()
    visiting = set()

    def run(name):
        if name in visiting:
            raise Exception("Cycle!")
        if name in done:
            return results[name]

        visiting.add(name)
        inputs = {}
        for dep in tasks[name]['deps']:
            inputs[dep] = run(dep)

        # inline: turn the string into a function right here
        f = tasks[name]['func']
        if callable(f):
            func = f
        else:
            ns = {}
            exec(f"_f = {f}", ns)
            func = ns['_f']

        results[name] = func(inputs)

        visiting.remove(name)
        done.add(name)
        return results[name]

    for name in tasks:
        run(name)
    return results

In [63]:
import ast
import types

def execute_dag(tasks):
    results = {}
    done = set()
    visiting = set()

    def make_func(f):
        if callable(f):
            return f
        # parse the string into an AST, compile it, build a function object
        module = ast.parse(f, mode='eval')
        code_obj = compile(module, '<string>', 'eval')
        # the compiled lambda's code is the first constant in the code object
        lambda_code = code_obj.co_consts[0]
        return types.FunctionType(lambda_code, {})

    def run(name):
        if name in visiting:
            raise Exception("Cycle!")
        if name in done:
            return results[name]

        visiting.add(name)
        inputs = {}
        for dep in tasks[name]['deps']:
            inputs[dep] = run(dep)

        func = make_func(tasks[name]['func'])
        results[name] = func(inputs)

        visiting.remove(name)
        done.add(name)
        return results[name]

    for name in tasks:
        run(name)
    return results

In [7]:
def execute_dag(tasks):
    results = {}

    while len(results) < len(tasks):
        print('data-1',results,tasks)
        for name, task in tasks.items():
            print(name,tasks)
            if name in results:
                continue                          # already done, skip
            if not all(d in results for d in task['deps']):
                continue                          # deps not ready, skip
            inputs = {d: results[d] for d in task['deps']}
            results[name] = eval(task['func'])(inputs)

    return results

In [8]:
tasks = {
    "extract":   {"deps": [],            "func": "lambda inputs: [1, 2, 3]"},
    "transform": {"deps": ["extract"],   "func": "lambda inputs: [x * 2 for x in inputs['extract']]"},
    "load":      {"deps": ["transform"], "func": "lambda inputs: sum(inputs['transform'])"}
}

print(execute_dag(tasks))
# {'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}

data-1 {} {'extract': {'deps': [], 'func': 'lambda inputs: [1, 2, 3]'}, 'transform': {'deps': ['extract'], 'func': "lambda inputs: [x * 2 for x in inputs['extract']]"}, 'load': {'deps': ['transform'], 'func': "lambda inputs: sum(inputs['transform'])"}}
extract {'extract': {'deps': [], 'func': 'lambda inputs: [1, 2, 3]'}, 'transform': {'deps': ['extract'], 'func': "lambda inputs: [x * 2 for x in inputs['extract']]"}, 'load': {'deps': ['transform'], 'func': "lambda inputs: sum(inputs['transform'])"}}
transform {'extract': {'deps': [], 'func': 'lambda inputs: [1, 2, 3]'}, 'transform': {'deps': ['extract'], 'func': "lambda inputs: [x * 2 for x in inputs['extract']]"}, 'load': {'deps': ['transform'], 'func': "lambda inputs: sum(inputs['transform'])"}}
load {'extract': {'deps': [], 'func': 'lambda inputs: [1, 2, 3]'}, 'transform': {'deps': ['extract'], 'func': "lambda inputs: [x * 2 for x in inputs['extract']]"}, 'load': {'deps': ['transform'], 'func': "lambda inputs: sum(inputs['transform']

In [9]:
from collections import deque

def execute_dag(tasks):
    results = {}

    # find tasks with no deps → start them first
    queue = deque([name for name, t in tasks.items() if not t['deps']])

    while queue:
        name = queue.popleft()

        # skip if deps not ready yet
        if not all(dep in results for dep in tasks[name]['deps']):
            queue.append(name)   # put back, try later
            continue

        inputs = {dep: results[dep] for dep in tasks[name]['deps']}
        results[name] = eval(tasks[name]['func'])(inputs)

        # add tasks that depend on this one
        for other, t in tasks.items():
            if name in t['deps'] and other not in results:
                queue.append(other)

    return results

In [10]:
tasks = {
    "extract":   {"deps": [],            "func": "lambda inputs: [1, 2, 3]"},
    "transform": {"deps": ["extract"],   "func": "lambda inputs: [x * 2 for x in inputs['extract']]"},
    "load":      {"deps": ["transform"], "func": "lambda inputs: sum(inputs['transform'])"}
}

print(execute_dag(tasks))

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [22]:
def execute_dag(tasks):
    # Step 1: figure out the order
    order = []
    visited = set()

    def topo(name):
        if name in visited:
            return
        visited.add(name)
        for dep in tasks[name]['deps']:   # do deps first
            topo(dep)
        order.append(name)               # add self after deps

    for name in tasks:
        topo(name)
        print(visited,name,order)
        

    # Step 2: run in order — deps always done before current
    results = {}
    for name in order:
        inputs = {dep: results[dep] for dep in tasks[name]['deps']}
        results[name] = eval(tasks[name]['func'])(inputs)
        print(inputs,results)

    return results

In [23]:
tasks = {
    "extract":   {"deps": [],            "func": "lambda inputs: [1, 2, 3]"},
    "transform": {"deps": ["extract"],   "func": "lambda inputs: [x * 2 for x in inputs['extract']]"},
    "load":      {"deps": ["transform"], "func": "lambda inputs: sum(inputs['transform'])"}
}


In [24]:
print(execute_dag(tasks))

{'extract'} extract ['extract']
{'extract', 'transform'} transform ['extract', 'transform']
{'load', 'extract', 'transform'} load ['extract', 'transform', 'load']
{} {'extract': [1, 2, 3]}
{'extract': [1, 2, 3]} {'extract': [1, 2, 3], 'transform': [2, 4, 6]}
{'transform': [2, 4, 6]} {'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}
{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [26]:
class TreeNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

# Build the tree
def build_tree():
    root = TreeNode(1)
    root.left        = TreeNode(2)
    root.right       = TreeNode(3)
    root.left.left   = TreeNode(4)
    root.left.right  = TreeNode(5)
    root.right.left  = TreeNode(6)
    root.right.right = TreeNode(7)
    return root


# ── Way 1: DFS Recursive ─────────────────────────────────────────
def search_dfs(node, target):
    if not node:
        return False                      # base case: not found
    if node.val == target:
        return True                       # found ✅
    return search_dfs(node.left, target) or search_dfs(node.right, target)


# ── Way 2: BFS (Level Order) ─────────────────────────────────────
from collections import deque

def search_bfs(root, target):
    if not root:
        return False
    queue = deque([root])

    while queue:
        node = queue.popleft()
        if node.val == target:
            return True                   # found ✅
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)

    return False


# ── Way 3: DFS Iterative (Stack) ─────────────────────────────────
def search_iterative(root, target):
    if not root:
        return False
    stack = [root]

    while stack:
        node = stack.pop()
        if node.val == target:
            return True                   # found ✅
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)

    return False


# ── Way 4: Return the NODE (not just True/False) ──────────────────
def find_node(node, target):
    if not node:
        return None
    if node.val == target:
        return node                       # return actual node ✅
    return find_node(node.left, target) or find_node(node.right, target)


# ── Way 5: Return PATH to node ────────────────────────────────────
def find_path(node, target, path=[]):
    if not node:
        return None
    path = path + [node.val]
    if node.val == target:
        return path                       # return full path ✅
    return find_path(node.left, target, path) or \
           find_path(node.right, target, path)


# ── Run all ───────────────────────────────────────────────────────
root = build_tree()
target = 6

print(search_dfs(root, target))        # True
print(search_bfs(root, target))        # True
print(search_iterative(root, target))  # True
print(find_node(root, target).val)     # 6
print(find_path(root, target))         # [1, 3, 6]

True
True
True
6
[1, 3, 6]


In [28]:
# Step 1: flatten tree to sorted array via inorder traversal
def inorder(node, result=[]):
    if not node:
        return
    inorder(node.left, result)    # left first
    result.append(node.val)       # then root
    inorder(node.right, result)   # then right
    return result

# Step 2: binary search on sorted array
def binary_search(arr, target):
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2

        if arr[mid] == target:
            return True           # ✅ found
        elif arr[mid] < target:
            left = mid + 1        # go right
        else:
            right = mid - 1       # go left

    return False                  # not found


# Run it
from collections import deque

class TreeNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

root = TreeNode(1)
root.left        = TreeNode(2)
root.right       = TreeNode(3)
root.left.left   = TreeNode(4)
root.left.right  = TreeNode(5)
root.right.left  = TreeNode(6)
root.right.right = TreeNode(7)

arr = inorder(root, [])
print("Sorted array:", arr)          # [4, 2, 5, 1, 6, 3, 7]

print(binary_search(arr, 6))         # True ✅
print(binary_search(arr, 9))         # False ❌

Sorted array: [4, 2, 5, 1, 6, 3, 7]
False
False


In [29]:
class BST:
    def __init__(self, val):
        self.val   = val
        self.left  = None
        self.right = None

def insert(root, val):
    if not root:
        return BST(val)
    if val < root.val:
        root.left  = insert(root.left, val)   # smaller → go left
    else:
        root.right = insert(root.right, val)  # larger  → go right
    return root

def bst_search(root, target):
    if not root:
        return False                          # not found
    if root.val == target:
        return True                           # ✅ found

    if target < root.val:
        return bst_search(root.left, target)  # go left
    else:
        return bst_search(root.right, target) # go right


# Build BST from values [1,2,3,4,5,6,7]
values = [4, 2, 6, 1, 3, 5, 7]   # insert 4 first as root for balance
bst_root = None
for v in values:
    bst_root = insert(bst_root, v)

#         4
#        / \
#       2   6
#      / \ / \
#     1  3 5  7   ← proper BST ✅

print(bst_search(bst_root, 6))    # True ✅
print(bst_search(bst_root, 9))    # False ❌

True
False


In [31]:
# 1
#            /     \
#           2       3
#          / \     / \
#         4   5   6   7
#        / \ / \
#       8  9 10 11

In [33]:
BST Rule: left < parent < right

Check:  4's left = 8,  4's right = 9
        8 > 4 ✅  9 > 4 ✅  but both on different sides
        
        2's left = 4, right = 5  ✅
        1's left = 2, right = 3  ✅ looks ok
        
BUT:    3's subtree has 6,7
        2's subtree has 4,5,8,9,10,11
        
        All of 2's subtree values > 3? ❌
        8,9,10,11 > 3 but sit on LEFT side of root
        
❌ NOT a BST → binary search won't work correctly

SyntaxError: invalid character '✅' (U+2705) (2535614727.py, line 4)

In [35]:
from collections import deque

class TreeNode:
    def __init__(self, val):
        self.val   = val
        self.left  = None
        self.right = None

# Build your exact tree
def build_tree():
    nodes = [TreeNode(i) for i in range(12)]  # 0 unused, 1-11

    nodes[1].left  = nodes[2];  nodes[1].right = nodes[3]
    nodes[2].left  = nodes[4];  nodes[2].right = nodes[5]
    nodes[3].left  = nodes[6];  nodes[3].right = nodes[7]
    nodes[4].left  = nodes[8];  nodes[4].right = nodes[9]
    nodes[5].left  = nodes[10]; nodes[5].right = nodes[11]

    return nodes[1]

# BFS — level by level search
def bfs_search(root, target):
    queue = deque([root])
    level = 0

    while queue:
        level_size = len(queue)
        level += 1
        print(f"Level {level}: ", end="")

        for _ in range(level_size):
            node = queue.popleft()
            print(node.val, end=" ")

            if node.val == target:
                print(f"\n✅ Found {target} at Level {level}!")
                return True

            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)

        print()

    print(f"❌ {target} not found")
    return False

root = build_tree()
bfs_search(root, 10)

Level 1: 1 
Level 2: 2 3 
Level 3: 4 5 6 7 
Level 4: 8 9 10 
✅ Found 10 at Level 4!


True

In [ ]:
def search_sorted(root, target):
    # Step 1: collect all values via BFS
    values = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        values.append(node.val)
        if node.left:  queue.append(node.left)
        if node.right: queue.append(node.right)

    values.sort()
    print("Sorted:", values)   # [1,2,3,4,5,6,7,8,9,10,11]

    # Step 2: binary search
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        print(f"  checking index {mid} → value {values[mid]}")

        if values[mid] == target:
            return True
        elif values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return False

print(search_sorted(root, 10))